In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import numpy as np
import joblib
import os

# Load and preprocess the data
df_ori = pd.read_csv(r'full_results_with_predictions.csv')

df_gc = pd.read_excel(r'synthetic_data_High_Low_moderate.xlsx')


# Define features and target
X_ori = df_ori[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID']].values
y_ori = df_ori['Chloride Ion Penetrability Class']

label_encoder = LabelEncoder()
y_ori = label_encoder.fit_transform(y_ori)

X_gc = df_gc[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID']].values
y_gc = df_gc['Chloride Ion Penetrability Class']
y_gc = label_encoder.fit_transform(y_gc)

# Step 1: Split the original data into training and testing sets
Xtrain, X_test, ytrain, y_test = train_test_split(X_ori, y_ori, train_size=0.8, random_state=42)

# Step 2: Concatenate original and synthetic datasets

# Convert NumPy arrays to pandas DataFrame/Series for concatenation
Xtrain_df = pd.DataFrame(Xtrain, columns=['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID'])
ytrain_df = pd.Series(ytrain, name='Chloride Ion Penetrability Class')

X_gc_df = pd.DataFrame(X_gc, columns=Xtrain_df.columns)
y_gc_df = pd.Series(y_gc, name='Chloride Ion Penetrability Class')


# Concatenate training data with synthetic data
X_train = pd.concat([Xtrain_df, X_gc_df], axis=0)
y_train = pd.concat([ytrain_df, y_gc_df], axis=0)

# At this point, X_train_combined and y_train_combined have your original and synthetic data ready


In [ ]:


# 1. Random Forest Classifier with Grid Search
rf_param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_report = classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_)

# 2. XGBoost Classifier with Grid Search
xgb_param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

xgb_grid = GridSearchCV(XGBClassifier(random_state=42), xgb_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
xgb_report = classification_report(y_test, y_pred_xgb, target_names=label_encoder.classes_)

# 3. MLP Classifier (ANN) with Grid Search
mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (150,)],
    'activation': ['relu', 'tanh'],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [300, 500]
}

mlp_grid = GridSearchCV(MLPClassifier(random_state=42), mlp_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
mlp_grid.fit(X_train, y_train)
best_mlp = mlp_grid.best_estimator_
y_pred_mlp = best_mlp.predict(X_test)
mlp_report = classification_report(y_test, y_pred_mlp, target_names=label_encoder.classes_)

# Displaying the results
print("Best Random Forest Classifier:")
print(f"Best Parameters: {rf_grid.best_params_}")
print(rf_report)

print("Best XGBoost Classifier:")
print(f"Best Parameters: {xgb_grid.best_params_}")
print(xgb_report)

print("Best MLP (ANN) Classifier:")
print(f"Best Parameters: {mlp_grid.best_params_}")
print(mlp_report)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Load and preprocess the data
df_ori = pd.read_csv(r'full_results_with_predictions.csv')
df_gc = pd.read_excel(r'synthetic_data_High_Low_moderate.xlsx')

# Define features and target
X_ori = df_ori[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID']].values
y_ori = df_ori['Chloride Ion Penetrability Class']

# Encoding target labels for original data
label_encoder = LabelEncoder()
y_ori = label_encoder.fit_transform(y_ori)

# Preprocess synthetic data
X_gc = df_gc[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID']].values
y_gc = df_gc['Chloride Ion Penetrability Class']
y_gc = label_encoder.fit_transform(y_gc)

# Step 1: Split the original data into training and testing sets
Xtrain, X_test, ytrain, y_test = train_test_split(X_ori, y_ori, train_size=0.8, random_state=42)

# Step 2: Concatenate original and synthetic datasets
Xtrain_df = pd.DataFrame(Xtrain, columns=['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID'])
ytrain_df = pd.Series(ytrain, name='Chloride Ion Penetrability Class')

X_gc_df = pd.DataFrame(X_gc, columns=Xtrain_df.columns)
y_gc_df = pd.Series(y_gc, name='Chloride Ion Penetrability Class')

# Concatenate training data with synthetic data
X_train = pd.concat([Xtrain_df, X_gc_df], axis=0)
y_train = pd.concat([ytrain_df, y_gc_df], axis=0)

# 1. Random Forest Classifier with Grid Search
rf_param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_report = classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_)

# 2. XGBoost Classifier with Grid Search
xgb_param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

xgb_grid = GridSearchCV(XGBClassifier(random_state=42), xgb_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
xgb_report = classification_report(y_test, y_pred_xgb, target_names=label_encoder.classes_)

# 3. MLP Classifier (ANN) with Grid Search
mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (150,)],
    'activation': ['relu', 'tanh'],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [300, 500]
}

mlp_grid = GridSearchCV(MLPClassifier(random_state=42), mlp_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
mlp_grid.fit(X_train, y_train)
best_mlp = mlp_grid.best_estimator_
y_pred_mlp = best_mlp.predict(X_test)
mlp_report = classification_report(y_test, y_pred_mlp, target_names=label_encoder.classes_)

# Displaying the results
print("Best Random Forest Classifier:")
print(f"Best Parameters: {rf_grid.best_params_}")
print(rf_report)

print("Best XGBoost Classifier:")
print(f"Best Parameters: {xgb_grid.best_params_}")
print(xgb_report)

print("Best MLP (ANN) Classifier:")
print(f"Best Parameters: {mlp_grid.best_params_}")
print(mlp_report)


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best Random Forest Classifier:
Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
              precision    recall  f1-score   support

        High       0.20      0.18      0.19        11
         Low       0.47      0.64      0.54        58
    Moderate       0.23      0.19      0.21        16
    Very Low       0.88      0.75      0.81       121

    accuracy                           0.65       206
   macro avg       0.44      0.44      0.44       206
weighted avg       0.67      0.65      0.65       206

Best XGBoost Classifier:
Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 300, 'subsample': 1}
              precision    recall  f1-score   support

        High       0.25      0.18      0.21        11
         Low       0.56      0.69      0.62        58
    Moderate       0.27      0.19      0.22        16
    Very Low       0.86      0.83      0.84       121

    accuracy   

/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# Load and preprocess the data
df_ori = pd.read_csv(r'full_results_with_predictions.csv')
df_gc = pd.read_excel(r'synthetic_data_High_Low_moderate.xlsx')

# Define features and target (assuming 'Chloride Ion Penetrability' is a continuous variable now)
X_ori = df_ori[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
                'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID']].values
y_ori = df_ori['COULOMB_TEST']  # Assuming continuous values

# Preprocess synthetic data
X_gc = df_gc[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
              'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID']].values
y_gc = df_gc['COULOMB_TEST']  # Assuming continuous values

# Step 1: Split the original data into training and testing sets
Xtrain, X_test, ytrain, y_test = train_test_split(X_ori, y_ori, train_size=0.8, random_state=42)

# Step 2: Concatenate original and synthetic datasets
Xtrain_df = pd.DataFrame(Xtrain, columns=['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'CAGG', 'WATER', 'AEA', 
                                          'WR_HR', 'WR', 'ACC', 'VOID', 'w/b', 'b/a', 'SCM%', 'APPLICATION_ID'])
ytrain_df = pd.Series(ytrain, name='COULOMB_TEST')

X_gc_df = pd.DataFrame(X_gc, columns=Xtrain_df.columns)
y_gc_df = pd.Series(y_gc, name='COULOMB_TEST')

# Concatenate training data with synthetic data
X_train = pd.concat([Xtrain_df, X_gc_df], axis=0)
y_train = pd.concat([ytrain_df, y_gc_df], axis=0)

# 1. Random Forest Regressor with Grid Search
rf_param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_param_grid, cv=3, n_jobs=-1, scoring='r2')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

# Calculate metrics for Random Forest
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)

# 2. XGBoost Regressor with Grid Search
xgb_param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

xgb_grid = GridSearchCV(XGBRegressor(random_state=42), xgb_param_grid, cv=3, n_jobs=-1, scoring='r2')
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

# Calculate metrics for XGBoost
xgb_r2 = r2_score(y_test, y_pred_xgb)
xgb_mse = mean_squared_error(y_test, y_pred_xgb)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)

# 3. MLP Regressor (ANN) with Grid Search
mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (150,)],
    'activation': ['relu', 'tanh'],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [300, 500]
}

mlp_grid = GridSearchCV(MLPRegressor(random_state=42), mlp_param_grid, cv=3, n_jobs=-1, scoring='r2')
mlp_grid.fit(X_train, y_train)
best_mlp = mlp_grid.best_estimator_
y_pred_mlp = best_mlp.predict(X_test)

# Calculate metrics for MLP Regressor
mlp_r2 = r2_score(y_test, y_pred_mlp)
mlp_mse = mean_squared_error(y_test, y_pred_mlp)
mlp_mae = mean_absolute_error(y_test, y_pred_mlp)

# Displaying the results
print("Best Random Forest Regressor:")
print(f"Best Parameters: {rf_grid.best_params_}")
print(f"R²: {rf_r2}")
print(f"MSE: {rf_mse}")
print(f"MAE: {rf_mae}\n")

print("Best XGBoost Regressor:")
print(f"Best Parameters: {xgb_grid.best_params_}")
print(f"R²: {xgb_r2}")
print(f"MSE: {xgb_mse}")
print(f"MAE: {xgb_mae}\n")

print("Best MLP (ANN) Regressor:")
print(f"Best Parameters: {mlp_grid.best_params_}")
print(f"R²: {mlp_r2}")
print(f"MSE: {mlp_mse}")
print(f"MAE: {mlp_mae}\n")


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/User

Best Random Forest Regressor:
Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500}
R²: 0.13025391435781075
MSE: 2244847.2725634864
MAE: 798.4614513317308

Best XGBoost Regressor:
Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
R²: 0.13851514059361292
MSE: 2223524.737757506
MAE: 854.8940174880537

Best MLP (ANN) Regressor:
Best Parameters: {'activation': 'relu', 'hidden_layer_sizes': (150,), 'learning_rate': 'constant', 'max_iter': 500}
R²: 0.20205750177582993
MSE: 2059519.5199739488
MAE: 791.1708285627811



/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(
